In [5]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

#loading data
file_path = r"C:\Users\gamer\Downloads\PS_2026.04.09_22.16.00.csv"

df = pd.read_csv(file_path, comment='#', low_memory=False)
df.columns = df.columns.str.strip().str.lower()
print(f"Initial Shape: {df.shape}")

#standardize column names
df = df.rename(columns={
    'discoverymethod':   'disc_method',
    'discoveryyear':     'disc_year',
    'discovery_year':    'disc_year',
    'discoveryfacility': 'disc_facility'
})

#selecting featuers from the dataset(as per specified in hackathon)
required_cols = [
    'pl_name', 'disc_method', 'disc_year', 'disc_facility',
    'pl_orbper', 'pl_rade', 'pl_masse', 'pl_eqt',
    'st_teff', 'st_rad', 'st_mass', 'sy_dist'
]
df = df[[col for col in required_cols if col in df.columns]].copy()
print(f"Columns selected: {df.columns.tolist()}")

#convert numberic type and remove placeholders
num_list = [
    'disc_year', 'pl_orbper', 'pl_rade', 'pl_masse',
    'pl_eqt', 'st_teff', 'st_rad', 'st_mass', 'sy_dist'
]
for col in num_list:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

# replace known NASA placeholder values with NAN
df.replace([-999, -9999, 9999], np.nan, inplace=True)

#aggregation

def safe_mode(x):
    m = x.mode()
    return m.iloc[0] if not m.empty else np.nan

num_cols = df.select_dtypes(include=np.number).columns.tolist()
cat_cols = [
    c for c in df.select_dtypes(include='object').columns
    if c not in ('pl_name', 'disc_method')
]

agg_dict = {col: 'median' for col in num_cols}
agg_dict.update({col: safe_mode for col in cat_cols})
if 'disc_method' in df.columns:
    agg_dict['disc_method'] = safe_mode

df = df.groupby('pl_name').agg(agg_dict).reset_index()
print(f"\nShape after aggregation (unique planets): {df.shape}")

#missing value report ans imputation

num_cols = df.select_dtypes(include=np.number).columns.tolist()

print("\nMissing value % per column (post-aggregation):")
missing_report = (df[num_cols].isnull().mean() * 100).round(1).sort_values(ascending=False)
print(missing_report)

cols_to_drop = []
for col in num_cols:
    missing_pct = df[col].isnull().mean()
    if missing_pct < 0.5:
        df[col] = df[col].fillna(df[col].median())
    else:
        cols_to_drop.append(col)

if cols_to_drop:
    print(f"\nDropping columns with >50% missing values: {cols_to_drop}")
    df = df.drop(columns=cols_to_drop)
else:
    print("\nNo columns dropped due to missing values.")

#clean data set with planets name

clean_df = df.copy()
clean_df.to_csv(r"C:\Users\gamer\Downloads\clean_with_names.csv", index=False)
print("\nSaved: clean_with_names.csv")

#ml ready dataset
df = df.dropna(subset=['disc_method']).reset_index(drop=True)

# class balance check critical before modeling
print("Class Distribution (disc_method):")
counts = df['disc_method'].value_counts()
print(counts)

imbalance_ratio = counts.max() / counts.min()
print(f"\nImbalance ratio (max/min): {imbalance_ratio:.0f}x")
if imbalance_ratio > 10:
    print("  Warning severe class imbalance detected!")
    print("  Use class_weight='balanced' in your classifier.")


rare_classes = counts[counts < 5].index.tolist()
if rare_classes:
    print(f"\n  Dropping rare classes with less than 5 samples: {rare_classes}")
    df = df[~df['disc_method'].isin(rare_classes)].reset_index(drop=True)
    print(f"  Shape after dropping rare classes: {df.shape}")

y = df['disc_method']
X = df.drop(columns=['pl_name', 'disc_method'], errors='ignore')

# cardinality reduction and encoding
if 'disc_facility' in X.columns:
    top_facilities = X['disc_facility'].value_counts().nlargest(10).index
    X['disc_facility'] = X['disc_facility'].where(
        X['disc_facility'].isin(top_facilities), 'Other'
    )

# drop_first=False preserves all meaningful facility categories
X = pd.get_dummies(X, drop_first=False)

# convert all bool columns to int (0/1) — required for ml model
bool_cols = [c for c in X.columns if X[c].dtype == bool]
X[bool_cols] = X[bool_cols].astype(int)
print(f"\nConverted {len(bool_cols)} bool columns to int (0/1).")

# report low-variance columns 
low_var_cols = [
    c for c in X.select_dtypes(include='number').columns
    if X[c].nunique() < 5
]
if low_var_cols:
    print(f"\nWarning Low-variance columns detected (may be over-imputed): {low_var_cols}")

#scaling

dummy_cols   = [c for c in X.columns if c.startswith('disc_facility_')]
numeric_cols = [c for c in X.columns if c not in dummy_cols]

scaler = StandardScaler()
X_scaled_num = pd.DataFrame(
    scaler.fit_transform(X[numeric_cols]),
    columns=numeric_cols
)

# combine scaled numerics + unscaled dummies (already 0/1, no need to scale)
X_scaled = pd.concat(
    [X_scaled_num, X[dummy_cols].reset_index(drop=True)],
    axis=1
)

#train-test split
min_class_count = y.value_counts().min()
stratify_param  = y if min_class_count >= 2 else None

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y,
    test_size=0.2,
    random_state=42,
    stratify=stratify_param
)
print(f"\nTrain size: {X_train.shape}  |  Test size: {X_test.shape}")

#saving ml ready dataset
final_df = pd.concat([X_scaled, y.reset_index(drop=True)], axis=1)

# ensure any remaining bool columns are int before saving
bool_cols_final = [c for c in final_df.columns if final_df[c].dtype == bool]
final_df[bool_cols_final] = final_df[bool_cols_final].astype(int)

final_df.to_csv(r"C:\Users\gamer\Downloads\ml_ready_data.csv", index=False)
print("\nSaved: ml_ready_data.csv")

print("FINAL DATASET SUMMARY")
print(f"Shape         : {final_df.shape}")
print(f"Features      : {X_scaled.shape[1]}")
print(f"Target classes: {final_df['disc_method'].nunique()}")
print(f"Null values   : {final_df.isnull().sum().sum()}")
print(f"Bool columns  : {len(bool_cols_final)} remaining (should be 0)")
print("\nColumn dtypes:")
print(final_df.dtypes)

Initial Shape: (39554, 289)
Columns selected: ['pl_name', 'disc_method', 'disc_year', 'disc_facility', 'pl_orbper', 'pl_rade', 'pl_masse', 'pl_eqt', 'st_teff', 'st_rad', 'st_mass', 'sy_dist']

Shape after aggregation (unique planets): (6158, 12)

Missing value % per column (post-aggregation):
pl_masse     61.7
pl_eqt       25.5
pl_rade      25.3
st_rad       10.3
st_teff       7.9
pl_orbper     5.6
sy_dist       2.1
st_mass       0.7
disc_year     0.0
dtype: float64

Dropping columns with >50% missing values: ['pl_masse']

Saved: clean_with_names.csv
Class Distribution (disc_method):
disc_method
Transit                          4523
Radial Velocity                  1180
Microlensing                      278
Imaging                            94
Transit Timing Variations          40
Eclipse Timing Variations          17
Orbital Brightness Modulation       9
Pulsar Timing                       8
Astrometry                          6
Pulsation Timing Variations         2
Disk Kinematics  